# Naive RAG Demo

This notebook tests the RAG pipeline with Cloud documents

## Imports

In [ ]:
import sys
sys.path.append('../')

from langchain_ollama import ChatOllama

from src.loaders.pdf_loader import PDFLoader
from src.preprocessing.cleaners import TextCleaner
from src.preprocessing.text_splitter import TextSplitter
from src.embeddings.embedder import Embedder
from src.vectorstores.faiss_store import FAISSStore
from src.rag.retriever import Retriever
from src.llm.generator import Generator
from src.rag.pipeline import Pipeline

## Components Initialization

In [3]:
# llm
llm = ChatOllama(model="qwen3.5:9b")

# AWS Loader
pdf_loader_aws = PDFLoader("../data/raw/cloud_docs/aws/")


# Preprocessing
cleaner = TextCleaner()
splitter = TextSplitter(chunk_size=500, chunk_overlap=50, csv_chunk_size=100)

# Embedder
embedder = Embedder(model_name="nomic-ai/nomic-embed-text-v1.5")

# Vector Store
vectorstore = FAISSStore(collection_name="cloudmind", embedding_dim=768)

# Retriever and Generator
retriever = Retriever(embedder=embedder, vectorstore=vectorstore)
generator = Generator(llm=llm)

Loading weights: 100%|██████████| 112/112 [00:00<00:00, 6110.57it/s]


## Pipeline

In [4]:
pipeline = Pipeline(
    loaders=[pdf_loader_aws],
    cleaner=cleaner,
    splitter=splitter,
    embedder=embedder,
    vectorstore=vectorstore,
    retriever=retriever,
    generator=generator,
    storage_path="../data/processed/faiss"
)

pipeline.build()

Loaded 842 documents
Cleaned 842 documents
Split into 4145 chunks


Batches: 100%|██████████| 130/130 [12:38<00:00,  5.83s/it]


Embedded 4145 chunks
Indexed 4145 chunks
Vector store saved to ../data/processed/faiss


## Testing pipeline

In [5]:
query = "What are the best practices for cost optimization in a multi-cloud architecture?"
response = pipeline.ask(query, k=5)

print(f"Question: {query}\n")
print(f"Answer:\n{response}")

Retrieved 5 chunks
Question: What are the best practices for cost optimization in a multi-cloud architecture?

Answer:
Based on the provided context, I cannot answer the question regarding best practices for a **multi-cloud architecture**.

The available documentation (AWS — wellarchitected-financial-services-industry-lens.pdf) focuses exclusively on **AWS**-specific strategies. While it recommends organizing an overall environment with a **multi-account strategy** and utilizing **AWS Billing and Cost Management** tools, it does not contain specific guidance or best practices for managing cost optimization across multiple cloud providers (e.g., AWS, Azure, GCP).

**Available AWS-focused strategies in the context include:**
*   **Audit-based cost optimization:** Using manual analysis or tools (AWS Billing and Cost Management, AWS Partner tools).
*   **Multi-account strategy:** Organizing your overall AWS environment (snippet [3]).
*   **Automation:** Automating time-consuming cloud oper

## Retrieved Chunks Inspection

In [7]:
chunks = retriever.retrieve(query, k=5)

for i, chunk in enumerate(chunks, 1):
    print(f"\n{'='*60}")
    print(f"Chunk {i}")
    print(f"Provider : {chunk.metadata.get('provider', 'unknown')}")
    print(f"Source   : {chunk.metadata.get('file_name', 'unknown')}")
    print(f"Score    : {chunk.metadata.get('similarity_score', 0):.4f}")
    print(f"Content  : {chunk.content[:1000]}...")


Chunk 1
Provider : aws
Source   : wellarchitected-financial-services-industry-lens.pdf
Score    : 0.8020
Content  : program to optimize them over time afterwards.
Investing the right amount of e ort in a cost optimization strategy up front allows you to realize 
the economic bene ts of the cloud more readily, by ensuring a consistent adherence to best 
practices and avoiding unnecessary over provisioning. The following sections provide techniques 
and best practices for the initial and ongoing implementation of Cloud Financial Management and 
cost optimization for your workloads...

Chunk 2
Provider : aws
Source   : wellarchitected-financial-services-industry-lens.pdf
Score    : 0.7777
Content  : audit-based cost optimization on existing cloud workloads
There is a process to examine existing cloud spend, and identify cost optimization opportunities 
using manual analysis, or the use of tools (AWS Billing and Cost Management and Cost 
Management and Cost Management tools, AWS Partner t